# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### 1. Method choice and why

I chose Logistic Regression as the first machine-learning model for the Content Refresh Opportunity Scoring lane.

The prepared dataset provides an observed binary outcome, `is_declining_label`, where 1 means the page is classified as declining and 0 means it is not. Logistic Regression fits this yes/no outcome and produces a probability that can be used to rank pages.

This fits the lane because the goal is prioritization rather than simply assigning a class. I will therefore use the model's predicted probability of decline as the ranking score and compare its ranking with the Week-4 rule-based baseline.

Logistic Regression is also a simple and interpretable starting point. It allows the model's strongest features to be inspected instead of adding complexity before knowing whether a learned model improves on the baseline.


## 2. Split design

I will use a client-grouped train/test split using `client_id` as the grouping variable.

This is appropriate because the dataset contains multiple content pages from the same client. A random row-level split could place pages from the same client in both training and testing, which could make the test result look better than performance on unseen clients.

The `client_id` column will only be used to create the split and will not be used as a model feature. The test clients will be completely held out from model training.

I will use a fixed random seed so that the split and results are reproducible.


In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

DATA_PATH = "/content/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

# Remove rows where avg_position means "no position data"
model_df = df[df["avg_position"] > 0].copy()

print("Rows after removing avg_position == 0:", len(model_df))
print("Clients:", model_df["client_id"].nunique())

# Grouped train/test split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        groups=model_df["client_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("\nTrain shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

print("\nClient overlap:")
overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(overlap)

print("\nTrain declining rate:")
print(train_df["is_declining_label"].mean())

print("\nTest declining rate:")
print(test_df["is_declining_label"].mean())

Rows after removing avg_position == 0: 28795
Clients: 31

Train shape: (22974, 45)
Test shape: (5821, 45)

Train clients: 24
Test clients: 7

Client overlap:
set()

Train declining rate:
0.5706886045094455

Test declining rate:
0.5399415907919601


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### 3A.Preparing the model features

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

MODEL_NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

train_df = train_df.copy()
test_df = test_df.copy()

# Numeric preparation
numeric_fill_zero = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

for column in numeric_fill_zero:
    train_df[column] = (
        pd.to_numeric(train_df[column], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    test_df[column] = (
        pd.to_numeric(test_df[column], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

# Categorical preparation
for column in MODEL_CATEGORICAL_FEATURES:
    train_df[column] = (
        train_df[column]
        .fillna("unknown")
        .astype(str)
        .replace({"": "unknown", "nan": "unknown"})
    )

    test_df[column] = (
        test_df[column]
        .fillna("unknown")
        .astype(str)
        .replace({"": "unknown", "nan": "unknown"})
    )

# Create log features
for frame in [train_df, test_df]:
    frame["log_impressions_90d"] = np.log1p(frame["impressions_90d"])
    frame["log_clicks_90d"] = np.log1p(frame["clicks_90d"])
    frame["log_sessions_90d"] = np.log1p(frame["sessions_90d"])
    frame["log_ai_sessions_90d"] = np.log1p(frame["ai_sessions_90d"])

X_train = train_df[
    MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
]

y_train = train_df["is_declining_label"]

X_test = test_df[
    MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
]

y_test = test_df["is_declining_label"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (22974, 26)
X_test: (5821, 26)
y_train: (22974,)
y_test: (5821,)


### 3B. Training

In [25]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, MODEL_NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, MODEL_CATEGORICAL_FEATURES)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['search_volume',
                                                   'competition', 'cpc',
                                                   'word_count', 'char_count',
                                                   'log_impressions_90d',
                                                   'log_clicks_90d',
                                                   'log_sessions_90d',
                                                   'log_ai_sessions_90d',
                                                   'days_with_impressions',
                                                   'days_with_sessio...
                                                   'ai_traffic_pct']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['competition_level',
                                                   'content_type',
                                                   'main_intent', 'age_tier',
                                                   'freshness_tier',
                                                   'word_count_tier',
                                                   'impression_tier',
                                                   'position_tier'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])

### 3C. Generatinng decline probabilities/ranking


In [26]:
test_df["model_score"] = model.predict_proba(X_test)[:, 1]

print(
    test_df[
        ["content_id", "client_id", "is_declining_label", "model_score"]
    ]
    .sort_values("model_score", ascending=False)
    .head(10)
    .to_string(index=False)
)

          content_id         client_id  is_declining_label  model_score
content_a928cb66d230 client_f369cb89fc                   1     0.927059
content_c82bc0c24241 client_f369cb89fc                   1     0.917234
content_7be5f150dc65 client_f369cb89fc                   0     0.915627
content_87c007fb5c26 client_f369cb89fc                   1     0.906835
content_b5e9e6453511 client_f369cb89fc                   1     0.904743
content_5d77d3077984 client_f369cb89fc                   1     0.904498
content_e5f459e737b7 client_f369cb89fc                   1     0.900971
content_d274ac4158ef client_4e07408562                   0     0.895458
content_823ea9b9b355 client_f369cb89fc                   1     0.894358
content_96dba8ca02c1 client_f369cb89fc                   1     0.894336


### 3D. Calculating the Week-4 baseline on test data

In [31]:

from sklearn.preprocessing import MinMaxScaler

# Work only with the held-out Week-5 test set
baseline_test = test_df.copy()

baseline_features = [
    "impressions_90d",
    "avg_position",
    "days_since_last_update"
]

# Normalize the three baseline signals within the test set
scaler = MinMaxScaler()

baseline_test[
    ["imp_norm", "pos_norm", "stale_norm"]
] = scaler.fit_transform(
    baseline_test[baseline_features]
)

# Higher score = higher refresh priority
baseline_test["baseline_score"] = (
      0.50 * baseline_test["imp_norm"]
    + 0.30 * baseline_test["pos_norm"]
    + 0.20 * baseline_test["stale_norm"]
)

# Show the highest-ranked pages from the Week-4 baseline
baseline_top10 = (
    baseline_test[
        [
            "content_id",
            "client_id",
            "is_declining_label",
            "baseline_score"
        ]
    ]
    .sort_values("baseline_score", ascending=False)
    .head(10)
)

print("Top 10 Week-4 Baseline Rankings:")
print(baseline_top10.to_string(index=False))

print("\nBaseline score range:")
print(
    f"Minimum: {baseline_test['baseline_score'].min():.4f}"
)
print(
    f"Maximum: {baseline_test['baseline_score'].max():.4f}"
)

print("\nBaseline test rows:", len(baseline_test))

Top 10 Week-4 Baseline Rankings:
          content_id         client_id  is_declining_label  baseline_score
content_5fe46e04994d client_4e07408562                   1        0.558586
content_8c19996aa890 client_4e07408562                   1        0.502676
content_4c36c775b818 client_4e07408562                   1        0.457861
content_9532f197bbc8 client_4e07408562                   1        0.354500
content_db5989a78dd3 client_4e07408562                   0        0.347707
content_661e1745db72 client_e29c9c180c                   0        0.308152
content_23f1cc8851a9 client_e29c9c180c                   0        0.279019
content_7275a6a3a8eb client_e29c9c180c                   0        0.256339
content_71a31b831092 client_e29c9c180c                   0        0.250821
content_15fe075b97bc client_4ec9599fc2                   0        0.244277

Baseline score range:
Minimum: 0.0016
Maximum: 0.5586

Baseline test rows: 5821


### 3D. Calculating the Week-4 baseline on test data

To make the comparison fair, I apply the Week-4 rule to the same held-out test clients used for Logistic Regression. The baseline uses the same three signals from Week 4: `impressions_90d`, `avg_position`, and `days_since_last_update`.

The baseline score is a weighted ranking score, with higher scores indicating higher refresh priority. No label-derived fields such as `trend_direction` or `trend_pct` are used in the baseline.


In [32]:
k_values = [10, 20, 50, 100]

comparison_rows = []

for model_name, score_column in [
    ("Week-4 Baseline", "baseline_score"),
    ("Logistic Regression", "model_score")
]:
    row = {"Model": model_name}

    for k in k_values:
        row[f"Precision@{k}"] = precision_at_k_from_df(
            baseline_test,
            score_column,
            k
        )

    comparison_rows.append(row)

comparison_table = pd.DataFrame(comparison_rows)

print(comparison_table.to_string(index=False))

              Model  Precision@10  Precision@20  Precision@50  Precision@100
    Week-4 Baseline           0.4           0.4          0.38           0.30
Logistic Regression           0.8           0.8          0.88           0.83


## 4. Errors and interpretation

The Logistic Regression model made both false-positive and false-negative errors when a 0.5 probability threshold was used for error inspection. There were **1,334 false positives** and **876 false negatives** in the held-out test set. This threshold is used only to inspect classification errors; the Content Refresh Opportunity Scoring lane uses the model probability as a ranking score.

Some errors were high-confidence errors. For example, one non-declining page received a model score of **0.916**, while one actually declining page received a score of only **0.050**. This shows that the learned ranking signal is useful but not perfect.

The model leaned most strongly on **log_impressions_90d**, **word_count_tier_1000-2000**, **freshness_tier_0-30**, **log_clicks_90d**, and **word_count**. For example, `log_impressions_90d` had the largest positive coefficient (**0.859**), while `log_clicks_90d` had a negative coefficient (**-0.527**). These coefficients represent associations learned from the observed training data, not causal effects.

The strongest features are plausible signals for content-performance analysis because visibility, clicks, content characteristics, freshness, and search position can be associated with whether a page is declining. However, the model does not establish that changing any of these factors would cause a page to decline or recover.

The target `is_declining_label` is defined from the observed `trend_direction`. The model does not use `trend_direction` or `trend_pct` as input features because they directly define or reveal the target and would create leakage. Instead, the model uses other available content, search, engagement, freshness, and position signals.

Overall, Logistic Regression produced a stronger ranking signal than the Week-4 rule-based baseline on this held-out client split. The improvement was measured consistently across the tested ranking cutoffs, with Precision@10 increasing from **0.40 to 0.80**, Precision@20 from **0.40 to 0.80**, Precision@50 from **0.38 to 0.88**, and Precision@100 from **0.30 to 0.83**.

These are measured results on this particular held-out client split and should be treated as decision-support evidence rather than a guarantee of future performance. Individual pages can still be ranked incorrectly, so the model should support content review rather than make automatic refresh decisions.


### 4A. Error analysis

In [33]:
error_analysis = baseline_test.copy()

error_analysis["actual"] = error_analysis["is_declining_label"]

# Predicted label using 0.5 probability threshold
error_analysis["predicted"] = (
    error_analysis["model_score"] >= 0.5
).astype(int)

# Identify errors
error_analysis["error_type"] = np.where(
    (error_analysis["actual"] == 0) & (error_analysis["predicted"] == 1),
    "False Positive",
    np.where(
        (error_analysis["actual"] == 1) & (error_analysis["predicted"] == 0),
        "False Negative",
        "Correct"
    )
)

# False positives: model is confident but actual label is 0
false_positives = (
    error_analysis[
        error_analysis["error_type"] == "False Positive"
    ]
    .sort_values("model_score", ascending=False)
)

print("Top False Positives:")
print(
    false_positives[
        [
            "content_id",
            "client_id",
            "is_declining_label",
            "model_score"
        ]
    ].head(10).to_string(index=False)
)

print("\nNumber of False Positives:", len(false_positives))


# False negatives: actual label is 1 but model gives lower score
false_negatives = (
    error_analysis[
        error_analysis["error_type"] == "False Negative"
    ]
    .sort_values("model_score", ascending=True)
)

print("\nTop False Negatives:")
print(
    false_negatives[
        [
            "content_id",
            "client_id",
            "is_declining_label",
            "model_score"
        ]
    ].head(10).to_string(index=False)
)

print("\nNumber of False Negatives:", len(false_negatives))

Top False Positives:
          content_id         client_id  is_declining_label  model_score
content_7be5f150dc65 client_f369cb89fc                   0     0.915627
content_d274ac4158ef client_4e07408562                   0     0.895458
content_7fa63804b8f1 client_4e07408562                   0     0.883818
content_1d2233dc3323 client_f369cb89fc                   0     0.877360
content_5d5653c4eb4f client_4e07408562                   0     0.860187
content_374e795aab68 client_f369cb89fc                   0     0.858990
content_72a51b4538b7 client_f369cb89fc                   0     0.854471
content_b84a5df88090 client_f369cb89fc                   0     0.854107
content_26d48a980581 client_f369cb89fc                   0     0.852796
content_412f4cf7d443 client_f369cb89fc                   0     0.852723

Number of False Positives: 1334

Top False Negatives:
          content_id         client_id  is_declining_label  model_score
content_13bbd72aea33 client_e29c9c180c                   1  

### 4B. What does Logistic Regression lean on?

In [34]:
# Section 4B: Logistic Regression feature interpretation

logistic_model = model

# Get feature names from the preprocessing pipeline
feature_names = logistic_model.named_steps["preprocessor"].get_feature_names_out()

# Get coefficients
coefficients = logistic_model.named_steps["classifier"].coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "absolute_coefficient": np.abs(coefficients)
})

feature_importance = feature_importance.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Top 15 Logistic Regression features:")
print(
    feature_importance[
        ["feature", "coefficient"]
    ].head(15).to_string(index=False)
)

Top 15 Logistic Regression features:
                                     feature  coefficient
                numeric__log_impressions_90d     0.859241
      categorical__word_count_tier_1000-2000     0.596902
            categorical__freshness_tier_0-30     0.572647
                     numeric__log_clicks_90d    -0.526938
                         numeric__word_count     0.503933
            categorical__freshness_tier_181+    -0.496707
            categorical__impression_tier_low     0.458239
       categorical__main_intent_navigational    -0.437715
                       numeric__avg_position    -0.401600
   categorical__content_type_keyword article     0.382783
categorical__content_type_comparison article    -0.363175
                  categorical__age_tier_365+    -0.359238
      categorical__word_count_tier_2000-3500    -0.351074
      categorical__impression_tier_excellent    -0.295921
                 categorical__age_tier_31-90     0.268278


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.